# Music Rating System (Data Analysis)

Para ejecutar el cuaderno, tener cargados un archivo `pX.csv` (donde `X` es el puntaje 1–10) para cada puntaje.


## Parte 1 - Análisis Básico

In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# Imports
import os, re, glob, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")

# Para reproducibilidad
np.random.seed(551)

In [ ]:
# Normalizar nombres de columnas
RAW_COLS_RENAME = {
    'Song': 'song',
    'Artist': 'artist',
    'Popularity': 'popularity',
    'BPM': 'bpm',
    'Genres': 'genres',
    'Parent Genres': 'parent_genres',
    'Album': 'album',
    'Album Date': 'album_date',
    'Time': 'track_time',
    'Dance': 'dance',
    'Energy': 'energy',
    'Acoustic': 'acoustic',
    'Instrumental': 'instrumental',
    'Happy': 'happy',
    'Speech': 'speech',
    'Live': 'live',
    'Loud (Db)': 'loud_db',
    'Key': 'key_raw',
    'Time Signature': 'time_signature',
    'Added At': 'added_at',
    'Spotify Track Id': 'spotify_track_id',
    'Album Label': 'album_label',
    'Camelot': 'camelot',
    'ISRC': 'isrc',
}

NUM_COLS = ['popularity','bpm','dance','energy','acoustic','instrumental','happy','speech','live','loud_db']


def infer_score_from_filename(path: str) -> int:
    """Infiere el puntaje a partir de un nombre tipo pX.csv,
     donde X es el número que indica el puntaje."""
    base = os.path.basename(path)
    m = re.match(r"p(\d+)\.csv$", base, re.IGNORECASE)
    if not m:
        return np.nan
    return int(m.group(1))

def parse_mmss_to_seconds(s: str) -> float:
    """Convierte 'MM:SS' o 'HH:MM:SS' a segundos (float). Devuelve NaN si no parsea."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if not s:
        return np.nan
    try:
        parts = [int(p) for p in s.split(':')]
        if len(parts) == 2:
            m, sec = parts
            return m*60 + sec
        elif len(parts) == 3:
            h, m, sec = parts
            return h*3600 + m*60 + sec
        else:
            return np.nan
    except Exception:
        return np.nan

def tidy_genres(col: pd.Series) -> pd.Series:
    """Normaliza géneros."""
    def _split(x):
        if pd.isna(x) or str(x).strip()=='' or str(x).strip().lower()=='nan':
            return []
        return [g.strip().lower() for g in str(x).split(',') if g.strip()]
    return col.apply(_split)

def tidy_artists(col: pd.Series) -> pd.Series:
    """Normaliza artistas: string -> lista de artistas (split por coma, trim; preserva casing)."""
    def _split(x):
        if pd.isna(x) or str(x).strip()=='' or str(x).strip().lower()=='nan':
            return []
        # Nota: NO partimos por '&' porque hay dúos como 'Dimitri Vegas & Like Mike'
        return [a.strip() for a in str(x).split(',') if a.strip()]
    return col.apply(_split)

def load_all_csv(pattern: str = 'p*.csv') -> pd.DataFrame:
    """ Procesa los archivos csv."""
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError("No se encontraron archivos 'p*.csv' en el directorio actual.")

    frames = []
    for f in files:
        try:
            df = pd.read_csv(f)
        except Exception as e:
            print(f"⚠️ Error leyendo {f}: {e}")
            continue
        # Renombrar columnas si existen
        df = df.rename(columns={k: v for k, v in RAW_COLS_RENAME.items() if k in df.columns})
        # Agregar score
        df['score'] = infer_score_from_filename(f)
        # Tipos básicos
        for c in NUM_COLS:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors='coerce')
        # Fecha álbum / agregado
        if 'album_date' in df.columns:
            df['album_date'] = pd.to_datetime(df['album_date'], errors='coerce')
            df['album_year'] = df['album_date'].dt.year
        if 'added_at' in df.columns:
            df['added_at'] = pd.to_datetime(df['added_at'], errors='coerce')
        # Duración a segundos
        if 'track_time' in df.columns:
            df['track_seconds'] = df['track_time'].apply(parse_mmss_to_seconds)
        # Géneros a listas + principal (primero)
        if 'genres' in df.columns:
            df['genres_list'] = tidy_genres(df['genres'])
            df['primary_genre'] = df['genres_list'].apply(lambda lst: lst[0] if lst else np.nan)
        if 'parent_genres' in df.columns:
            df['parent_genres_list'] = tidy_genres(df['parent_genres'])
            df['parent_primary_genre'] = df['parent_genres_list'].apply(lambda lst: lst[0] if lst else np.nan)
        # Artistas como lista + artista principal y cantidad
        if 'artist' in df.columns:
            # normalizar blancos
            df['artist'] = df['artist'].astype(str).str.strip()
            df['artists_list'] = tidy_artists(df['artist'])
            df['primary_artist'] = df['artists_list'].apply(lambda lst: lst[0] if lst else np.nan)
            df['n_artists'] = df['artists_list'].apply(len)
        # Normalizar artista/álbum/canción
        for c in ['album','song','album_label','key_raw','camelot']:
            if c in df.columns:
                df[c] = df[c].astype(str).str.strip()
        frames.append(df)

    full = pd.concat(frames, ignore_index=True, sort=False)
    # Quitar filas sin score
    full = full[~full['score'].isna()].copy()

    # Columnas útiles reordenadas
    preferred_cols = [
        'song','artist','artists_list','primary_artist','n_artists',
        'album','album_year','album_date','added_at','score',
        'popularity','bpm','dance','energy','acoustic','instrumental','happy','speech','live','loud_db',
        'track_seconds','genres','primary_genre','parent_genres','parent_primary_genre',
        'camelot','key_raw','time_signature','spotify_track_id','album_label','isrc'
    ]
    cols = [c for c in preferred_cols if c in full.columns] + [c for c in full.columns if c not in preferred_cols]
    return full[cols]

# Carga
df = load_all_csv()
print(f"✅ Canciones cargadas: {len(df):,}")
df.head()

In [ ]:
# Valores faltantes por columna
na_summary = df.isna().mean().sort_values(ascending=False)
print("% de NA por columna (top 15):")
print((na_summary*100).head(15).round(1))


# Duplicados potenciales por (song, artist, album)
dups = df.duplicated(subset=[c for c in ['song','artist','album'] if c in df.columns], keep=False)
print(f"\nPosibles duplicados: {dups.sum():,}")

In [ ]:
# Ver duplicados
df[dups].sort_values(by=['song','artist','album'], ignore_index=True)

In [ ]:
min_tracks_artist = 60 # Umbral para determinar si un artista realmente es individualmente analizable

# Explota la lista de artistas: una fila por artista participante
df_any = (df.explode('artists_list', ignore_index=True)
            .rename(columns={'artists_list': 'artist_any'}))

g_artist = (df_any.dropna(subset=['artist_any'])
                 .groupby('artist_any', dropna=False)
                 .agg(n_tracks   = ('song', 'count'),
                      avg_score  = ('score', 'mean'))
                 .sort_values(['avg_score','n_tracks'], ascending=[False, False]))

print(f"Top artistascon mínimo de {min_tracks_artist} temas:")
print(g_artist.query('n_tracks >= @min_tracks_artist').head(10))

In [ ]:
min_tracks_album = 5 # Cuántos temas tiene para ser considerado album?
cols_album_group = [c for c in ['artist','album'] if c in df.columns]
if cols_album_group:
    g_album = (df.groupby(cols_album_group, dropna=False)
                 .agg(n_tracks=('song','count'),
                      avg_score=('score','mean'))
                 .sort_values(['avg_score','n_tracks'], ascending=[False, False]))
    print("Top álbumes por promedio (mínimo de", min_tracks_album, "):")
    print(g_album.query('n_tracks >= @min_tracks_album').head(10))
else:
    print("No hay columnas suficientes para agrupar por álbum.")

In [ ]:
if 'genres_list' in df.columns:
    exploded = df.explode('genres_list', ignore_index=True)
    g_gen = (exploded.dropna(subset=['genres_list'])
             .groupby('genres_list')
             .agg(n_tracks=('song','count'), avg_score=('score','mean'))
             .sort_values(['avg_score','n_tracks'], ascending=[False, False]))
    min_tracks_genre = 60
    print("Top géneros por promedio (mínimo de", min_tracks_genre, "):")
    print(g_gen.query('n_tracks >= @min_tracks_genre').head(20))
else:
    print("No hay columna 'genres_list' para explotar géneros.")

## Visualizaciones básicas - Punrajes

In [ ]:
# Histograma de puntajes
plt.figure(figsize=(6,3.5))
sns.countplot(x='score', data=df, color=sns.color_palette('Set2')[0])
plt.title('Distribución de puntajes')
plt.xlabel('Puntaje')
plt.ylabel('Cantidad de canciones')
plt.tight_layout()
plt.show()

# Boxplot por género padre (top 8 por volumen)
if 'parent_primary_genre' in df.columns:
    top_parents = (df['parent_primary_genre']
                   .value_counts().head(8).index.tolist())
    plt.figure(figsize=(8,4))
    sns.boxplot(x='parent_primary_genre', y='score', data=df[df['parent_primary_genre'].isin(top_parents)],
                order=top_parents)
    plt.title('Distribución de puntajes por género padre (top por volumen)')
    plt.xlabel('Género padre')
    plt.ylabel('Puntaje')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

# Relación BPM vs Puntaje
if 'bpm' in df.columns:
    plt.figure(figsize=(6,4))
    sns.regplot(x='bpm', y='score', data=df, scatter_kws={'alpha':0.4, 's':25}, line_kws={'color':'red'})
    plt.title('BPM vs Puntaje')
    plt.xlabel('BPM')
    plt.ylabel('Puntaje')
    plt.tight_layout()
    plt.show()


In [ ]:
# Promedio de puntaje por año del álbum
if 'album_year' in df.columns:
    yearly = (df.dropna(subset=['album_year'])
                .assign(album_year=lambda d: d['album_year'].astype(int))
                .groupby('album_year', as_index=False)
                .agg(n=('score','size'), avg_score=('score','mean'))
                .sort_values('album_year'))

    fig, ax = plt.subplots(figsize=(8,4))
    ax2 = ax.twinx()

    # Barras (cantidad) al eje primario
    ax.bar(yearly['album_year'], yearly['n'], color='#4b77f1', alpha=0.35, width=0.8, label='Cantidad', zorder=1)
    ax.set_ylabel('Cantidad de canciones', color='#4b77f1')
    ax.tick_params(axis='y', labelcolor='#4b77f1')

    # Línea (puntaje) al eje secundario
    sns.lineplot(data=yearly, x='album_year', y='avg_score', marker='o', color='#1f77b4', ax=ax2, zorder=2)
    ax2.set_ylabel('Puntaje promedio', color='#1f77b4')
    ax2.tick_params(axis='y', labelcolor='#1f77b4')

    ax.set_xlabel('Año del álbum')
    ax.set_title('Evolución del puntaje promedio por año de álbum')
    ax.grid(axis='y', alpha=0.2); ax2.grid(False)
    plt.tight_layout(); plt.show()
else:
    print("No hay 'album_year' para tendencia temporal.")

## Parte 2 - Variables

In [ ]:
# === EXPLICABILIDAD DE PUNTAJES: correlaciones + modelos + importancias ===
sns.set_theme(style="whitegrid")

# 1) Selección de features numéricas
num_cols = [c for c in [
    'bpm','dance','energy','acoustic','instrumental','happy','speech','live',
    'loud_db','track_seconds','popularity','album_year'
] if c in df.columns]

In [ ]:
# 2) Target encoding simple para categorías (géneros)
cat_cols = [c for c in ['primary_genre','parent_primary_genre'] if c in df.columns]
df_enc = df.copy()

for c in cat_cols:
    if df_enc[c].notna().sum()>0:
        means = df_enc.groupby(c)['score'].mean()
        df_enc[c+'_te'] = df_enc[c].map(means)  # promedio de tu score por categoría
    else:
        df_enc[c+'_te'] = np.nan

enc_cols = [c+'_te' for c in cat_cols if c+'_te' in df_enc.columns]

In [ ]:
# 3) Dataset final
X = df_enc[num_cols + enc_cols].copy()
y = df_enc['score'].astype(float)
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]

In [ ]:
# 4) Correlaciones (Spearman = robusto a no linealidades monótonas)
corr = X.apply(lambda s: s.corr(y, method='spearman')).sort_values(ascending=False)

plt.figure(figsize=(6,3.8))
sns.barplot(x=corr.values, y=corr.index, palette='viridis')
plt.title('Correlación (Spearman) con tu puntaje')
plt.xlabel('Spearman ρ'); plt.ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
# 5) Modelos: Lasso (lineal, con regularización) + Random Forest (no lineal)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=551)

# Lasso con estandarización
scaler = StandardScaler()
Xs_train = scaler.fit_transform(X_train)
Xs_test  = scaler.transform(X_test)
lasso = LassoCV(cv=5, random_state=42).fit(Xs_train, y_train)
lasso_coef = pd.Series(lasso.coef_, index=X.columns).sort_values(key=np.abs, ascending=False)

# Random Forest
rf = RandomForestRegressor(n_estimators=600, random_state=42, n_jobs=-1).fit(X_train, y_train)

In [ ]:
# 6) Permutation Importance en test (reduce sesgo del modelo)
pi = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=42)
perm_imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=True)

fig, axes = plt.subplots(1,2, figsize=(11,4), sharey=False)
# Lasso (magnitud de coeficientes)
lasso_plot = lasso_coef.head(10)[::-1]
axes[0].barh(lasso_plot.index, lasso_plot.values, color='#1f77b4')
axes[0].set_title('Importancia (|coef|) - Lasso')
axes[0].set_xlabel('Magnitud |coef|')

# RF Permutation importance
perm_plot = perm_imp.tail(10)
axes[1].barh(perm_plot.index, perm_plot.values, color='#2ca02c')
axes[1].set_title('Permutation Importance - Random Forest')
axes[1].set_xlabel('ΔR² promedio al permutar')

plt.tight_layout(); plt.show()

In [ ]:
# 7) Métricas rápidas
from sklearn.metrics import r2_score, mean_absolute_error
print(f"Lasso  -> R² test: {r2_score(y_test, lasso.predict(Xs_test)):.3f} | MAE: {mean_absolute_error(y_test, lasso.predict(Xs_test)):.3f}")
print(f"RF     -> R² test: {r2_score(y_test, rf.predict(X_test)):.3f} | MAE: {mean_absolute_error(y_test, rf.predict(X_test)):.3f}")

In [ ]:
# 8) Top explicadores resumidos
summary = pd.DataFrame({
    'spearman_r': corr,
    'lasso_|coef|': lasso_coef.abs(),
    'rf_perm_imp': pd.Series(pi.importances_mean, index=X.columns)
}).fillna(0)

print("\nTop 10 variables más consistentes (suma de ranks en 3 criterios):")
ranked = (summary.rank(ascending=False).sum(axis=1).sort_values().head(10))
display(summary.loc[ranked.index].sort_values('rf_perm_imp', ascending=False).round(3))

In [ ]:
# %% [code]
# === Partial Dependence & ICE: happy, acoustic, loud_db, primary_genre_te ===
import numpy as np, matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

# Asegurar modelo y datos (si no existen por alguna razón)
try:
    rf
    X_test
except NameError:
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestRegressor
    # Si no existen X, y de la sección anterior, levantamos num + enc rápido
    num_cols = [c for c in [
        'bpm','dance','energy','acoustic','instrumental','happy','speech','live',
        'loud_db','track_seconds','popularity','album_year'
    ] if c in df.columns]
    cat_cols = [c for c in ['primary_genre','parent_primary_genre'] if c in df.columns]
    df_enc = df.copy()
    for c in cat_cols:
        if df_enc[c].notna().sum()>0:
            means = df_enc.groupby(c)['score'].mean()
            df_enc[c+'_te'] = df_enc[c].map(means)
    enc_cols = [c+'_te' for c in cat_cols if c+'_te' in df_enc.columns]
    X = df_enc[num_cols + enc_cols].dropna()
    y = df_enc.loc[X.index, 'score'].astype(float)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    rf = RandomForestRegressor(n_estimators=600, random_state=42, n_jobs=-1).fit(X_train, y_train)

# Features objetivo (filtradas por disponibilidad)
feat_targets = [f for f in ['happy','acoustic','loud_db','primary_genre_te'] if f in X_test.columns]
if not feat_targets:
    raise ValueError("Ninguna de las features objetivo está en X_test.")

# PDP + ICE 1D
n = len(feat_targets)
rows = int(np.ceil(n/2))
fig, axes = plt.subplots(rows, 2, figsize=(10, 3.2*rows))
axes = np.array(axes).reshape(-1)  # a plano

PartialDependenceDisplay.from_estimator(
    rf, X_test, features=feat_targets, kind='both',  # 'average' + 'individual'
    percentiles=(0.02, 0.98), grid_resolution=30,
    ax=axes[:n],
    pd_line_kw={'color':'#1f77b4', 'lw':2},
    ice_lines_kw={'color':'#1f77b4', 'alpha':0.15, 'lw':1}
)
for ax in axes[n:]:
    ax.remove()
fig.suptitle('Dependencia parcial (promedio) + ICE (trayectorias individuales)', y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

# PDP 2D de interacción (si existen happy & acoustic)
if all(f in feat_targets for f in ['happy','acoustic']):
    fig, ax = plt.subplots(figsize=(5.6,4.4))
    PartialDependenceDisplay.from_estimator(
        rf, X_test, features=[('happy','acoustic')], kind='average',
        percentiles=(0.02, 0.98), grid_resolution=30,
        ax=ax,
    )
    ax.set_title('Interacción 2D: happy × acoustic (PDP)')
    plt.tight_layout(); plt.show()

## Parte 3 - Predictor